# 04 — Real-Time Implementation

Implementasi klasifikasi latihan secara real-time menggunakan webcam, MediaPipe Pose, dan model CNN-BiLSTM.

In [ ]:
import os
import sys
import cv2
import time
import mediapipe as mp
import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque, Counter
from scipy.ndimage import gaussian_filter1d
from tensorflow.keras.models import load_model


#KONFIGURASI MODEL DAN ROBUST SCALER
MODEL_PATH = 'assets/kasus7_murni_best.keras'
SCALER_MEDIAN_PATH = 'assets/scaler_median_7.npy'
SCALER_IQR_PATH = 'assets/scaler_iqr_7.npy'

CLASSES_LIST = ["jumpingjack", "lunges", "pushups", "squat", "Situp", "idle"]
SEQUENCE_LENGTH = 30
CONFIDENCE_THRESHOLD = 0.65
MOTION_THRESHOLD = 0.010

COLORS = {
    'jumpingjack': (0, 165, 255),
    'lunges': (0, 255, 255),
    'pushups': (0, 255, 0),
    'squat': (255, 0, 255),
    'Situp': (0, 100, 255),
    'idle': (200, 200, 200),
    'OUT_OF_FRAME': (0, 0, 255),
    'MENGANALISIS...': (255, 255, 0),
    'BUFFERING': (255, 150, 0)
}

LOG_DIR = 'output/logs'
PLOT_DIR = 'output/plots'



#LOGIKA DASAR & EKSTRAKSI FITUR (BIOMEKANIK)
def check_visibility(landmarks):
    shoulder_vis = (landmarks[11].visibility + landmarks[12].visibility) / 2
    hip_vis = (landmarks[23].visibility + landmarks[24].visibility) / 2
    knee_vis = (landmarks[25].visibility + landmarks[26].visibility) / 2
    return shoulder_vis > 0.5 and hip_vis > 0.5 and knee_vis > 0.5


def calculate_motion_score(window_data):
    # Pantau pergerakan Bahu (1), Pinggul (19), Lutut (25), dan Engkel (31)
    # Menentukan apakah model diinferensi atatu tidak
    return np.mean([
        np.std(window_data[:, 1]),  # BAHU
        np.std(window_data[:, 19]),  # Pinggul
        np.std(window_data[:, 25]),  # Lutut
        np.std(window_data[:, 31])  # Engkel
    ])


def calculate_angle(a, b, c):
    #Menghitung sudut antara tiga titik (a, b, c) dengan b sebagai titik pusat.
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))


def extract_features_45(landmarks):
    """
    Mengekstraksi 45 fitur dari 12 landmark utama MediaPipe.
    Input: 33 landmark MediaPipe
    Output: 45 fitur (36 koordinat mentah + 9 fitur geometris)
    """
    indices = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
    frame_raw, xs, ys = [], [], []
    for idx in indices:
        lm = landmarks[idx]
        frame_raw.extend([lm.x, lm.y, lm.z])
        xs.append(lm.x);
        ys.append(lm.y)
    frame_raw = np.array(frame_raw)

    def get_pt(idx): return np.array([frame_raw[idx], frame_raw[idx + 1]])

    l_sh, r_sh = get_pt(0), get_pt(3)
    l_hip, r_hip = get_pt(18), get_pt(21)
    l_knee, r_knee = get_pt(24), get_pt(27)
    l_ank, r_ank = get_pt(30), get_pt(33)

    ankle_dist = np.abs(frame_raw[30] - frame_raw[33])
    knee_ver_diff = np.abs(frame_raw[25] - frame_raw[28])

    ang_l_knee = calculate_angle(l_hip, l_knee, l_ank)
    ang_r_knee = calculate_angle(r_hip, r_knee, r_ank)
    ang_l_hip = calculate_angle(l_sh, l_hip, l_knee)
    ang_r_hip = calculate_angle(r_sh, r_hip, r_knee)

    mid_shoulder = (l_sh + r_sh) / 2
    mid_hip = (l_hip + r_hip) / 2
    trunk_vec = mid_shoulder - mid_hip
    norm_trunk = np.linalg.norm(trunk_vec) + 1e-8
    dot_product = np.dot(trunk_vec, np.array([0, -1]))
    feat_trunk = np.degrees(np.arccos(np.clip(dot_product / norm_trunk, -1.0, 1.0))) / 180.0

    feat_knee_sym = np.abs(ang_l_knee - ang_r_knee) / 180.0
    width, height = max(xs) - min(xs), max(ys) - min(ys)
    feat_ratio = min(height / (width + 0.001), 3.0) / 3.0
    norm_angles = [ang_l_knee / 180.0, ang_r_knee / 180.0, ang_l_hip / 180.0, ang_r_hip / 180.0]

    return np.concatenate(
        [frame_raw, [ankle_dist, knee_ver_diff], norm_angles, [feat_trunk, feat_knee_sym, feat_ratio]])



#INFERENSI MODEL(CNN-LSTM)
class ExerciseModel:
    def __init__(self):
        self.model = load_model(MODEL_PATH, compile=False)
        self.scaler_median = np.load(SCALER_MEDIAN_PATH).squeeze()
        self.scaler_iqr = np.load(SCALER_IQR_PATH).squeeze()

    def compute_135_features(self, window, smooth_sigma=0.3):
        if smooth_sigma > 0:
            smoothed = np.zeros_like(window)
            for j in range(window.shape[1]):
                smoothed[:, j] = gaussian_filter1d(window[:, j], sigma=smooth_sigma)
        else:
            smoothed = window
        velocity = np.diff(smoothed, axis=0, prepend=smoothed[0:1])
        acceleration = np.diff(velocity, axis=0, prepend=velocity[0:1])
        return np.concatenate([window, velocity, acceleration], axis=-1)

    def predict(self, sequence_data):
        window_135 = self.compute_135_features(sequence_data)
        window_norm = (window_135 - self.scaler_median) / (self.scaler_iqr + 1e-6)
        input_data = np.expand_dims(window_norm, axis=0)
        prediction = self.model.predict(input_data, verbose=0)
        max_idx = np.argmax(prediction[0])
        pred_conf = np.max(prediction[0])
        return max_idx, float(pred_conf)


#VISUALISASI DAN UI PROGRAM
CUSTOM_CONNECTIONS = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
    (11, 23), (12, 24), (23, 24), (23, 25), (25, 27),
    (24, 26), (26, 28)
]
USED_INDICES = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]


def draw_custom_skeleton(frame, landmarks):
    #Menggambar kerangka tubuh pada frame berdasarkan landmark yang terdeteksi.
    h, w, _ = frame.shape
    for connection in CUSTOM_CONNECTIONS:
        idx1, idx2 = connection
        lm1, lm2 = landmarks[idx1], landmarks[idx2]
        if lm1.visibility > 0.5 and lm2.visibility > 0.5:
            pt1 = (int(lm1.x * w), int(lm1.y * h))
            pt2 = (int(lm2.x * w), int(lm2.y * h))
            cv2.line(frame, pt1, pt2, (255, 255, 255), 3)

    for idx in USED_INDICES:
        lm = landmarks[idx]
        if lm.visibility > 0.5:
            pt = (int(lm.x * w), int(lm.y * h))
            cv2.circle(frame, pt, 6, (0, 0, 255), -1)


def draw_ui_klasifikasi(frame, status, fps, inf_time, mp_time, queue_size, waktu_berjalan):
    h, w, _ = frame.shape
    overlay = frame.copy()

    cv2.rectangle(overlay, (0, 0), (w, 95), (15, 15, 15), -1)
    cv2.rectangle(overlay, (0, h - 40), (w, h), (15, 15, 15), -1)
    frame = cv2.addWeighted(overlay, 0.8, frame, 0.2, 0)

    #Label Prediksi
    color = COLORS.get(status['class'], (255, 255, 255))
    main_text = "SIAP (IDLE)" if status['class'] == 'idle' else status['class'].upper()
    if status['class'] in ['OUT_OF_FRAME', 'MENGANALISIS...', 'BUFFERING']:
        main_text = status['message'] or status['class']

    cv2.putText(frame, "PREDIKSI AI:", (20, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    font_scale = 0.9 if len(main_text) > 15 else 1.1
    cv2.putText(frame, main_text, (20, 60), cv2.FONT_HERSHEY_DUPLEX, font_scale, color, 2)
    #Menampilkan Confidence Score
    if status['class'] not in ['OUT_OF_FRAME', 'MENGANALISIS...', 'BUFFERING', 'idle']:
        cv2.putText(frame, f"Conf: {status['confidence'] * 100:.1f}%", (320, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color,
                    2)
    #TIMER
    menit, detik = int(waktu_berjalan // 60), int(waktu_berjalan % 60)
    cv2.putText(frame, f"TIMER: {menit:02d}:{detik:02d}", (w // 2 - 100, 55), cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 0, 255),
                2)
    #Metrik performa (FPS, MediaPipe time,AI infer time)
    cv2.putText(frame, f"FPS       : {int(fps)}", (w - 280, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    cv2.putText(frame, f"MP Time   : {int(mp_time)} ms", (w - 280, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    cv2.putText(frame, f"AI Infer  : {int(inf_time)} ms", (w - 280, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255),
                1)

    cv2.circle(frame, (w - 80, 25), 8, (0, 0, 255), -1)
    cv2.putText(frame, "REC", (w - 65, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    if queue_size < SEQUENCE_LENGTH and status['class'] not in ['OUT_OF_FRAME']:
        cv2.rectangle(frame, (0, 95), (int(w * (queue_size / SEQUENCE_LENGTH)), 100), (255, 150, 0), -1)

    cv2.putText(frame, "Sistem HAR Full AI | 'S' Toggle Skeleton | 'Q' Keluar", (20, h - 15), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (200, 200, 200), 1)
    return frame


def generate_thesis_plots(csv_path):
    target_dir = os.path.dirname(csv_path)
    if not target_dir: target_dir = "."
    base_name = os.path.basename(csv_path).replace('.csv', '')

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f" Gagal membaca CSV untuk grafik: {e}")
        return

    sns.set_theme(style="whitegrid")

    # 1. TIMELINE PREDIKSI
    plt.figure(figsize=(12, 5))
    mapping = {'IDLE': 0, 'JUMPINGJACK': 1, 'SQUAT': 2, 'PUSHUPS': 3, 'LUNGES': 4, 'SITUP': 5, 'BUFFERING': 6,
               'OUT_OF_FRAME': 6}
    df['ID_Tebakan'] = df['Tebakan'].str.upper().map(mapping).fillna(6)
    plt.step(df['Waktu (Detik)'], df['ID_Tebakan'], where='post', color='#2c3e50', linewidth=2)
    plt.yticks(range(7), ['IDLE', 'JJ', 'SQUAT', 'PUSHUP', 'LUNGES', 'SITUP', 'OTHER'])
    plt.title(f'Timeline Prediksi Gerakan AI - {base_name}', fontsize=14)
    plt.xlabel('Waktu (Detik)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(target_dir, f"{base_name}_1_timeline.png"), dpi=300)
    plt.close()

    # 2. CONFIDENCE TREND
    plt.figure(figsize=(12, 4))
    plt.plot(df['Waktu (Detik)'], df['Confidence'], color='#e67e22', linewidth=1.5)
    plt.fill_between(df['Waktu (Detik)'], df['Confidence'], color='#e67e22', alpha=0.2)
    plt.ylim(0, 1.1)
    plt.title(f'Fluktuasi Confidence Level - {base_name}', fontsize=14)
    plt.ylabel('Confidence (0-1)')
    plt.tight_layout()
    plt.savefig(os.path.join(target_dir, f"{base_name}_2_confidence.png"), dpi=300)
    plt.close()

    # 3. FPS & RAM
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.set_xlabel('Waktu (Detik)')
    ax1.set_ylabel('FPS', color='#27ae60')
    ax1.plot(df['Waktu (Detik)'], df['FPS'], color='#27ae60', alpha=0.8)
    ax1.tick_params(axis='y', labelcolor='#27ae60')
    ax2 = ax1.twinx()
    ax2.set_ylabel('RAM Usage (MB)', color='#7f8c8d')
    ax2.fill_between(df['Waktu (Detik)'], df['RAM Usage (MB)'], color='#7f8c8d', alpha=0.2)
    ax2.tick_params(axis='y', labelcolor='#7f8c8d')
    plt.title(f'Analisis Performa Sistem - {base_name}', fontsize=14)
    fig.tight_layout()
    plt.savefig(os.path.join(target_dir, f"{base_name}_3_performance.png"), dpi=300)
    plt.close()

    # 4. LATENCY
    plt.figure(figsize=(8, 6))
    avg_mp = df['MediaPipe Time (ms)'].mean()
    avg_ai = df['AI Infer Time (ms)'].mean()
    plt.bar(['Latency'], [avg_mp], color='#3498db', label='MediaPipe')
    plt.bar(['Latency'], [avg_ai], bottom=[avg_mp], color='#e74c3c', label='AI LSTM')
    plt.ylabel('Waktu (ms)')
    plt.title(f'Distribusi Latency Komputasi - {base_name}', fontsize=14)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(target_dir, f"{base_name}_4_latency.png"), dpi=300)
    plt.close()

    print(f" 4 Grafik metrik otomatis berhasil disimpan di: {target_dir}")



#FUNGSI UTAMA (MAIN LOOP CAMERA)
def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')


def main():
    clear_screen()
    print("=" * 65)
    print(" SISTEM HAR SKELETON-BASED (MODE FULL AI)".center(65))
    print("=" * 65)
    nama_subjek = input("Masukkan Nama Subjek (contoh: Andi_Testing): ")
    if not nama_subjek: nama_subjek = "Subjek_Test"

    # Setup Folder Penyimpanan
    base_dir = "Data_Pengujian"
    subject_dir = os.path.join(base_dir, nama_subjek)
    os.makedirs(subject_dir, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    file_prefix = f"FullAI_{timestamp}"
    csv_filename = os.path.join(subject_dir, f"{file_prefix}.csv")
    video_filename = os.path.join(subject_dir, f"{file_prefix}.mp4")

    print(f"\n Penyimpanan: {subject_dir}")
    print("Memuat AI Model CNN-LSTM...")

    # Inisialisasi Model AI
    ai_model = ExerciseModel()

    # Inisialisasi MediaPipe
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

    # Inisialisasi Kamera
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    frame_width, frame_height = 1280, 720
    cap.set(cv2.CAP_PROP_FPS, 30)
    fps_kamera = 30.0

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video = cv2.VideoWriter(video_filename, fourcc, fps_kamera, (frame_width, frame_height))

    frames_queue = deque(maxlen=SEQUENCE_LENGTH)
    PREDICTION_HISTORY = deque(maxlen=9)
    log_performa = []

    current_status = {'class': 'BUFFERING', 'confidence': 0.0, 'message': 'MENGUMPULKAN DATA...'}
    prev_frame_time, frame_count = 0, 0
    show_skeleton = True
    process_memory = psutil.Process(os.getpid())

    window_name = f"Testing AI: {nama_subjek}"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 1024, 768)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        frame_count += 1
        new_frame_time = time.time()
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Proses MediaPipe
        t_mp_start = time.time()
        results = pose.process(rgb_frame)
        mp_time_ms = (time.time() - t_mp_start) * 1000

        inference_time_ms = 0
        ram_usage_mb = process_memory.memory_info().rss / (1024 * 1024)
        cpu_usage_percent = process_memory.cpu_percent(interval=None)

        # Variabel Biomekanik Default
        sudut_lutut_kiri, sudut_lutut_kanan = 180.0, 180.0
        sudut_siku_kiri, sudut_siku_kanan = 180.0, 180.0
        sudut_pinggul_kiri = 180.0
        jarak_ankle_x, tinggi_vertikal, aspect_ratio_log = 0.0, 0.0, 0.0

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            if show_skeleton: draw_custom_skeleton(frame, landmarks)

            # Ekstraksi Biomekanik untuk kebutuhan Log CSV Skripsi
            l_hip = [landmarks[23].x, landmarks[23].y, landmarks[23].z]
            l_knee = [landmarks[25].x, landmarks[25].y, landmarks[25].z]
            l_ank = [landmarks[27].x, landmarks[27].y, landmarks[27].z]
            r_hip = [landmarks[24].x, landmarks[24].y, landmarks[24].z]
            r_knee = [landmarks[26].x, landmarks[26].y, landmarks[26].z]
            r_ank = [landmarks[28].x, landmarks[28].y, landmarks[28].z]

            sudut_lutut_kiri = calculate_angle(l_hip, l_knee, l_ank)
            sudut_lutut_kanan = calculate_angle(r_hip, r_knee, r_ank)

            l_shldr = [landmarks[11].x, landmarks[11].y, landmarks[11].z]
            l_elbow = [landmarks[13].x, landmarks[13].y, landmarks[13].z]
            l_wrist = [landmarks[15].x, landmarks[15].y, landmarks[15].z]
            r_shldr = [landmarks[12].x, landmarks[12].y, landmarks[12].z]
            r_elbow = [landmarks[14].x, landmarks[14].y, landmarks[14].z]
            r_wrist = [landmarks[16].x, landmarks[16].y, landmarks[16].z]

            sudut_siku_kiri = calculate_angle(l_shldr, l_elbow, l_wrist)
            sudut_siku_kanan = calculate_angle(r_shldr, r_elbow, r_wrist)
            sudut_pinggul_kiri = calculate_angle(l_shldr, l_hip, l_knee)
            jarak_ankle_x = abs(landmarks[27].x - landmarks[28].x)

            mid_sh_y = (landmarks[11].y + landmarks[12].y) / 2
            mid_hip_y = (landmarks[23].y + landmarks[24].y) / 2
            tinggi_vertikal = mid_hip_y - mid_sh_y

            xs = [landmarks[i].x for i in [11, 12, 23, 24, 25, 26, 27, 28]]
            ys = [landmarks[i].y for i in [11, 12, 23, 24, 25, 26, 27, 28]]
            aspect_ratio_log = (max(ys) - min(ys)) / ((max(xs) - min(xs)) + 1e-6)

            # Logika Deteksi
            if not check_visibility(landmarks):
                current_status = {'class': 'OUT_OF_FRAME', 'confidence': 0.0,
                                  'message': 'Pastikan Seluruh Tubuh Terlihat'}
                frames_queue.clear()
            else:
                try:
                    feat_45 = extract_features_45(landmarks)
                    frames_queue.append(feat_45)
                except:
                    pass

                if len(frames_queue) < SEQUENCE_LENGTH:
                    current_status = {'class': 'BUFFERING', 'confidence': 0.0, 'message': 'MENGUMPULKAN DATA...'}
                else:
                    window_data = np.array(frames_queue)

                    # Filter Gerakan (Motion Threshold)
                    if calculate_motion_score(window_data) < MOTION_THRESHOLD:
                        predicted_class, confidence = 'idle', 1.0
                    else:
                        t_start = time.time()
                        max_idx, pred_conf = ai_model.predict(window_data)
                        inference_time_ms = (time.time() - t_start) * 1000
                        raw_prediction = CLASSES_LIST[max_idx]

                        # Temporal Smoothing (Stabilisasi Prediksi)
                        PREDICTION_HISTORY.append(raw_prediction)
                        if len(PREDICTION_HISTORY) >= 9:
                            suara_olahraga = [p for p in PREDICTION_HISTORY if p != 'idle']
                            if len(suara_olahraga) >= 4:
                                predicted_class = Counter(suara_olahraga).most_common(1)[0][0]
                            else:
                                predicted_class = Counter(PREDICTION_HISTORY).most_common(1)[0][0]
                        else:
                            predicted_class = 'MENGANALISIS...'

                        confidence = pred_conf

                    if predicted_class != 'MENGANALISIS...':
                        current_status = {'class': predicted_class, 'confidence': confidence, 'message': ''}
        else:
            current_status = {'class': 'OUT_OF_FRAME', 'confidence': 0.0, 'message': 'Tidak Ada Orang!'}
            frames_queue.clear()

        # Proses Logging Data per Frame
        fps = 1 / (new_frame_time - prev_frame_time) if prev_frame_time != 0 else 0
        prev_frame_time = new_frame_time
        waktu_berjalan = frame_count / fps_kamera

        log_data = {
            "Frame": frame_count, "Waktu (Detik)": round(waktu_berjalan, 2), "FPS": round(fps, 2),
            "CPU Usage (%)": round(cpu_usage_percent, 2), "RAM Usage (MB)": round(ram_usage_mb, 2),
            "MediaPipe Time (ms)": round(mp_time_ms, 2), "AI Infer Time (ms)": round(inference_time_ms, 2),
            "Tebakan": current_status['class'], "Confidence": round(current_status['confidence'], 2),
            "Sudut_Lutut_Kiri": round(sudut_lutut_kiri, 1), "Sudut_Lutut_Kanan": round(sudut_lutut_kanan, 1),
            "Avg_Lutut": round((sudut_lutut_kiri + sudut_lutut_kanan) / 2, 1),
            "Sudut_Siku_Kiri": round(sudut_siku_kiri, 1), "Sudut_Siku_Kanan": round(sudut_siku_kanan, 1),
            "Sudut_Pinggul": round(sudut_pinggul_kiri, 1), "Jarak_Ankle_X": round(jarak_ankle_x, 3),
            "Tinggi_Vertikal": round(tinggi_vertikal, 3), "Aspect_Ratio": round(aspect_ratio_log, 3)
        }

        # Gambar UI ke layar
        frame = draw_ui_klasifikasi(frame, current_status, fps, inference_time_ms, mp_time_ms, len(frames_queue),
                                    waktu_berjalan)

        log_performa.append(log_data)
        out_video.write(frame)
        cv2.imshow(window_name, frame)

        # Kontrol Keyboard
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s'):
            show_skeleton = not show_skeleton

    # Cleanup Memory & Save Data
    cap.release()
    out_video.release()
    cv2.destroyAllWindows()

    # Simpan log dan hasilkan grafik
    if len(log_performa) > 0:
        df_log = pd.DataFrame(log_performa)
        try:
            df_log.to_csv(csv_filename, index=False)
            print(f"\nData log skripsi disimpan di: {csv_filename}")
            # Generate 4 Plot Metrik
            generate_thesis_plots(csv_filename)
        except Exception as e:
            print(f"Gagal menyimpan data: {e}")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\nAplikasi dihentikan.")
        sys.exit()